<a href="https://colab.research.google.com/github/vicfior/Projeto-DataSUS/blob/main/02_etl_SNIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# SPIKE SNIS — exploração inicial
# Objetivo: confirmar anos disponíveis, schema, granularidade
# ============================================================

# 1. Instalar a biblioteca
!pip install basedosdados -q

# 2. Importar
import basedosdados as bd
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 9.8 MB/s eta 0:00:00


In [ ]:
PROJECT_ID = "projeto-drsai"

# Query: quais anos existem e quantos municípios cobrem?
query_anos = """
SELECT
  ano,
  COUNT(DISTINCT id_municipio) AS n_municipios,
  COUNT(*) AS n_registros
FROM `basedosdados.br_mdr_snis.municipio_agua_esgoto`
WHERE ano BETWEEN 2017 AND 2024
GROUP BY ano
ORDER BY ano
"""

df_anos = bd.read_sql(query_anos, billing_project_id=PROJECT_ID)
print(df_anos)

Downloading: 100%|██████████|
    ano  n_municipios  n_registros
0  2017          5117         5117
1  2018          5136         5136
2  2019          5177         5177
3  2020          5338         5338
4  2021          5313         5313
5  2022          5425         5425


In [ ]:
# Schema completo da tabela
query_schema = """
SELECT column_name, data_type
FROM `basedosdados.br_mdr_snis.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'municipio_agua_esgoto'
ORDER BY ordinal_position
"""

df_schema = bd.read_sql(query_schema, billing_project_id=PROJECT_ID)
print(f"Total de colunas: {len(df_schema)}")
print(df_schema.to_string())

Downloading: 100%|██████████|
Total de colunas: 133
                                                column_name data_type
0                                                       ano     INT64
1                                              id_municipio    STRING
2                                                  sigla_uf    STRING
3                                   populacao_atendida_agua     INT64
4                                 populacao_atentida_esgoto     INT64
5                                          populacao_urbana     INT64
6                           populacao_urbana_residente_agua     INT64
7                            populacao_urbana_atendida_agua     INT64
8                       populacao_urbana_atendida_agua_ibge     INT64
9                         populacao_urbana_residente_esgoto     INT64
10                         populacao_urbana_atendida_esgoto     INT64
11                   populacao_urbana_residente_esgoto_ibge     INT64
12                                    

IN023 — Atendimento urbano água --> indice_atendimento_urbano_agua (63)\
IN015 — Coleta de esgoto --> indice_coleta_esgoto (57)\
IN046 — Esgoto tratado / água consumida --> indice_esgotamento_agua_consumida (69)

In [ ]:
COLUNAS_SNIS = [
    # Identificação
    "ano",
    "id_municipio",
    "sigla_uf",

    # Cobertura — hipótese: maior cobertura → menor risco DRSAI
    "indice_atendimento_total_agua",        # cobertura geral água
    "indice_coleta_esgoto",                 # IN015 oficial
    "indice_tratamento_esgoto",             # quanto do esgoto coletado é tratado

    # Qualidade — hipótese: perdas ≈ contaminação possível
    "indice_perda_distribuicao_agua",       # perdas físicas na rede

    # Adequação esgoto/água — hipótese: descompasso = esgoto não tratado
    "indice_esgotamento_agua_consumida",    # IN046 oficial
]

3 hipóteses metodológicas:

Cobertura insuficiente (água ou esgoto) → exposição direta a patógenos → maior
risco DRSAI\
Perdas na rede de água → pontos vulneráveis a contaminação → maior risco DRSAI\
Descompasso esgoto/água (esgoto coletado < água consumida) → esgoto a céu aberto → maior risco DRSAI



In [ ]:
PROJECT_ID = "projeto-drsai"

query_qualidade = """
SELECT
  ano,
  COUNT(*) AS n_municipios,

  -- Estatísticas dos 6 indicadores
  ROUND(AVG(indice_atendimento_total_agua), 2) AS media_atend_agua,
  ROUND(AVG(indice_atendimento_urbano_agua), 2) AS media_atend_urb_agua,
  ROUND(AVG(indice_coleta_esgoto), 2) AS media_coleta_esgoto,
  ROUND(AVG(indice_tratamento_esgoto), 2) AS media_trat_esgoto,
  ROUND(AVG(indice_perda_distribuicao_agua), 2) AS media_perdas,
  ROUND(AVG(indice_esgotamento_agua_consumida), 2) AS media_esg_consumo,

  -- % de municípios com cada feature preenchida
  ROUND(100 * COUNT(indice_coleta_esgoto) / COUNT(*), 1) AS pct_tem_coleta,
  ROUND(100 * COUNT(indice_tratamento_esgoto) / COUNT(*), 1) AS pct_tem_trat

FROM `basedosdados.br_mdr_snis.municipio_agua_esgoto`
WHERE ano BETWEEN 2017 AND 2022
GROUP BY ano
ORDER BY ano
"""

import basedosdados as bd
df_qualidade = bd.read_sql(query_qualidade, billing_project_id=PROJECT_ID)
print(df_qualidade.to_string())

Downloading: 100%|██████████|
    ano  n_municipios  media_atend_agua  media_atend_urb_agua  media_coleta_esgoto  media_trat_esgoto  media_perdas  media_esg_consumo  pct_tem_coleta  pct_tem_trat
0  2017          5117             69.02                 91.72                63.56              68.00         31.53              41.31            47.6          47.7
1  2018          5136             69.67                 91.90                63.76              69.05         32.52              42.27            48.9          49.0
2  2019          5177             70.09                 92.02                64.16              68.98         32.66              42.43            50.0          50.0
3  2020          5338             70.62                 92.26                63.70              66.80         32.30              40.93            52.4          52.5
4  2021          5313             71.18                 92.64                64.15              68.39         32.66              42.54           

Os indicadores de cobertura de saneamento apresentaram variação interanual média inferior a 1 p.p. no período 2017-2022 (n=6 anos, n=5.000+ municípios). Justifica-se assim a decisão metodológica de utilizar SNIS 2022 como proxy para os anos de modelagem 2023-2024.

In [ ]:
PROJECT_ID = "projeto-drsai"

query_diagnostico = """
SELECT
  ano,
  COUNT(*) AS total_municipios,
  COUNT(indice_atendimento_urbano_agua) AS com_dado,
  COUNT(*) - COUNT(indice_atendimento_urbano_agua) AS sem_dado
FROM `basedosdados.br_mdr_snis.municipio_agua_esgoto`
WHERE ano BETWEEN 2017 AND 2022
GROUP BY ano
ORDER BY ano
"""

df_diag = bd.read_sql(query_diagnostico, billing_project_id=PROJECT_ID)
print(df_diag)

Downloading: 100%|██████████|
    ano  total_municipios  com_dado  sem_dado
0  2017              5117      5115         2
1  2018              5136      5136         0
2  2019              5177      5177         0
3  2020              5338      5337         1
4  2021              5313      5302        11
5  2022              5425         0      5425


In [ ]:
query_2022 = """
SELECT
  COUNT(*) AS total,
  COUNT(indice_atendimento_total_agua) AS atend_total_agua,
  COUNT(indice_atendimento_urbano_agua) AS atend_urb_agua,
  COUNT(indice_coleta_esgoto) AS coleta_esgoto,
  COUNT(indice_tratamento_esgoto) AS trat_esgoto,
  COUNT(indice_perda_distribuicao_agua) AS perdas,
  COUNT(indice_esgotamento_agua_consumida) AS esg_consumo
FROM `basedosdados.br_mdr_snis.municipio_agua_esgoto`
WHERE ano = 2022
"""

df_2022 = bd.read_sql(query_2022, billing_project_id=PROJECT_ID)
print(df_2022.to_string())

Downloading: 100%|██████████|
   total  atend_total_agua  atend_urb_agua  coleta_esgoto  trat_esgoto  perdas  esg_consumo
0   5425              5424               0           2895         2897    5405         2895


2022 não tem o atend_urb_agua. Trocar por atend_total_agua

In [ ]:
INDICADORES_SNIS = [
    "indice_atendimento_total_agua",       # cobertura água (sempre preenchido)
    "indice_coleta_esgoto",                # IN015 — imputa 0 se NaN
    "indice_tratamento_esgoto",            # imputa 0 se NaN
    "indice_perda_distribuicao_agua",      # qualidade rede água
    "indice_esgotamento_agua_consumida",   # IN046 — imputa 0 se NaN
]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

PATH_SNIS_RAW = Path('/content/drive/MyDrive/projeto_drsai/data/raw/snis')
PATH_SNIS_RAW.mkdir(parents=True, exist_ok=True)
print(f"✅ Pasta pronta: {PATH_SNIS_RAW}")

✅ Pasta pronta: /content/drive/MyDrive/projeto_drsai/data/raw/snis


In [ ]:
PROJECT_ID = "projeto-drsai"

query_snis = """
SELECT
  ano,
  id_municipio,
  sigla_uf,
  populacao_atendida_agua,
  populacao_atentida_esgoto,
  indice_atendimento_total_agua,
  COALESCE(indice_coleta_esgoto, 0)              AS indice_coleta_esgoto,
  COALESCE(indice_tratamento_esgoto, 0)          AS indice_tratamento_esgoto,
  indice_perda_distribuicao_agua,
  COALESCE(indice_esgotamento_agua_consumida, 0) AS indice_esgotamento_agua_consumida
FROM `basedosdados.br_mdr_snis.municipio_agua_esgoto`
WHERE ano BETWEEN 2017 AND 2022
ORDER BY ano, id_municipio
"""

import basedosdados as bd
df_snis = bd.read_sql(query_snis, billing_project_id=PROJECT_ID)

caminho_saida = PATH_SNIS_RAW / "snis_municipio_2017_2022.parquet"
df_snis.to_parquet(caminho_saida, index=False)

print(f"✅ SNIS extraído: {len(df_snis):,} linhas")
print(f"   Anos: {sorted(df_snis['ano'].unique())}")
print(f"   Municípios únicos: {df_snis['id_municipio'].nunique()}")
print(f"   Salvo em: {caminho_saida}")
print(f"\nAmostra:")
print(df_snis.head(3).to_string())

Downloading: 100%|██████████|
✅ SNIS extraído: 31,506 linhas
   Anos: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
   Municípios únicos: 5493
   Salvo em: /content/drive/MyDrive/projeto_drsai/data/raw/snis/snis_municipio_2017_2022.parquet

Amostra:
    ano id_municipio sigla_uf  populacao_atendida_agua  populacao_atentida_esgoto  indice_atendimento_total_agua  indice_coleta_esgoto  indice_tratamento_esgoto  indice_perda_distribuicao_agua  indice_esgotamento_agua_consumida
0  2017      1100015       RO                    10675                       <NA>                          41.97                  0.00                       0.0                           61.66                               0.00
1  2017      1100023       RO                    86193                       1865                          80.30                  1.48                     100.0                           56.54                               1.48
2  2017      11

## Disponibilidade temporal do SNIS

O SNIS (Sistema Nacional de Informações sobre Saneamento) disponibiliza dados
até 2022 na Base dos Dados. Os anos de 2023-2024 estão em processo de migração
para o SINISA e ainda não foram publicados. A estratégia adotada foi **carry
forward de 2022**, justificada pela estabilidade dos indicadores (variação < 1
p.p./ano no período 2017-2022). Ver célula de justificativa abaixo.

 O dataset SNIS cobre os anos de 2017 a 2022 com dados reais de 5.117 a 5.425 municípios por ano. Para os anos de modelagem 2023-2024, ausentes do SNIS em razão da transição para o SINISA, foram criados registros por carry forward dos valores de 2022. A coluna dado_real diferencia explicitamente registros originais (n=31.506) de registros imputados (n=10.850). A validade dessa suposição é sustentada pela estabilidade observada nos indicadores entre 2017-2022, com variação média inferior a 1 p.p. por ano.

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

# Carrega o arquivo salvo
df_snis = pd.read_parquet(
    '/content/drive/MyDrive/projeto_drsai/data/raw/snis/snis_municipio_2017_2022.parquet'
)

# Carry forward: duplica 2022 para 2023 e 2024
df_2022 = df_snis[df_snis['ano'] == 2022].copy()

df_2023 = df_2022.copy()
df_2023['ano'] = 2023

df_2024 = df_2022.copy()
df_2024['ano'] = 2024

# Consolida
df_snis_completo = pd.concat([df_snis, df_2023, df_2024], ignore_index=True)

# Adiciona flag para identificar carry forward (transparência metodológica)
df_snis_completo['dado_real'] = df_snis_completo['ano'] <= 2022

# Salva versão completa
caminho_completo = Path('/content/drive/MyDrive/projeto_drsai/data/interim/snis')
caminho_completo.mkdir(parents=True, exist_ok=True)

df_snis_completo.to_parquet(
    caminho_completo / 'snis_municipio_2017_2024.parquet',
    index=False
)

print(f"✅ SNIS completo: {len(df_snis_completo):,} linhas")
print(f"   Anos: {sorted(df_snis_completo['ano'].unique())}")
print(f"   Reais: {df_snis_completo['dado_real'].sum():,}")
print(f"   Carry forward: {(~df_snis_completo['dado_real']).sum():,}")
print(f"\nDistribuição por ano:")
print(df_snis_completo.groupby('ano')['dado_real'].agg(['count','first']).rename(
    columns={'count':'municipios','first':'dado_real'}
))

✅ SNIS completo: 42,356 linhas
   Anos: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
   Reais: 31,506
   Carry forward: 10,850

Distribuição por ano:
      municipios  dado_real
ano                        
2017        5117       True
2018        5136       True
2019        5177       True
2020        5338       True
2021        5313       True
2022        5425       True
2023        5425      False
2024        5425      False


In [ ]:
import pandas as pd
from pathlib import Path

# Carrega SNIS completo e tabela de macrorregiões
df_snis = pd.read_parquet(
    '/content/drive/MyDrive/projeto_drsai/data/interim/snis/snis_municipio_2017_2024.parquet'
)

# Carrega tabela de macrorregiões (a mesma do notebook 02)
df_macro = pd.read_csv(
    '/content/drive/MyDrive/projeto_drsai/data/raw/macrorregioes_saude_ms_seidigi_2024.csv',
    dtype={"Codigo Municipio": "string"}
)

# Renomeia colunas da tabela de macrorregiões
df_macro = df_macro.rename(columns={
    "Codigo Municipio":             "cod_municipio",
    "Codigo Macrorregiao de Saude": "cod_macrorregiao",
    "Macrorregiao de Saude":        "nome_macrorregiao",
    "Populacao Estimada IBGE 2022": "populacao_2022",
}).assign(
    populacao_2022=lambda x: (
        x["populacao_2022"].astype(str)
         .str.replace(".", "", regex=False)
         .pipe(pd.to_numeric, errors="coerce")
    )
)[["cod_municipio", "cod_macrorregiao", "nome_macrorregiao", "populacao_2022"]]

# correção do código identificador do SNIS em relação ao SIH
df_snis["id_municipio"] = df_snis["id_municipio"].astype(str).str[:6]

# Merge SNIS com macrorregiões
df_merged = df_snis.merge(
    df_macro,
    left_on="id_municipio",
    right_on="cod_municipio",
    how="left"
)

# Diagnóstico do merge
n_sem_macro = df_merged["cod_macrorregiao"].isna().sum()
pct_sem_macro = 100 * n_sem_macro / len(df_merged)
print(f"Municípios sem macrorregião: {n_sem_macro} ({pct_sem_macro:.1f}%)")

# Remove os sem macrorregião (logging antes de descartar)
df_merged = df_merged.dropna(subset=["cod_macrorregiao"])

# Indicadores SNIS para agregar
INDICADORES = [
    "indice_atendimento_total_agua",
    "indice_coleta_esgoto",
    "indice_tratamento_esgoto",
    "indice_perda_distribuicao_agua",
    "indice_esgotamento_agua_consumida",
]

# Agregação por macrorregião × ano (média ponderada pela população 2022)
# Usa populacao_2022 como peso estável (não varia por ano — simplificação aceitável)
def media_ponderada(grupo, coluna, peso="populacao_2022"):
    """Média ponderada com tratamento de NaN no peso."""
    mask = grupo[coluna].notna() & grupo[peso].notna()
    if mask.sum() == 0:
        return float("nan")
    valores = grupo.loc[mask, coluna]
    pesos   = grupo.loc[mask, peso]
    return (valores * pesos).sum() / pesos.sum()

registros = []
for (macro, ano), grupo in df_merged.groupby(["cod_macrorregiao", "ano"]):
    reg = {
        "cod_macrorregiao": macro,
        "nome_macrorregiao": grupo["nome_macrorregiao"].iloc[0],
        "ano": ano,
        "dado_real_snis": grupo["dado_real"].all(),
        "n_municipios": len(grupo),
    }
    for ind in INDICADORES:
        reg[ind] = media_ponderada(grupo, ind)
    registros.append(reg)

df_snis_macro = pd.DataFrame(registros)

# Salva
caminho_saida = Path('/content/drive/MyDrive/projeto_drsai/data/processed')
caminho_saida.mkdir(parents=True, exist_ok=True)
df_snis_macro.to_parquet(
    caminho_saida / 'snis_macrorregiao_2017_2024.parquet',
    index=False
)

print(f"\n✅ SNIS por macrorregião: {len(df_snis_macro):,} linhas")
print(f"   Macrorregiões únicas: {df_snis_macro['cod_macrorregiao'].nunique()}")
print(f"   Anos: {sorted(df_snis_macro['ano'].unique())}")
print(f"\nAmostra:")
print(df_snis_macro.head(5).to_string(index=False))

Municípios sem macrorregião: 0 (0.0%)

✅ SNIS por macrorregião: 968 linhas
   Macrorregiões únicas: 121
   Anos: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Amostra:
 cod_macrorregiao          nome_macrorregiao  ano  dado_real_snis  n_municipios  indice_atendimento_total_agua  indice_coleta_esgoto  indice_tratamento_esgoto  indice_perda_distribuicao_agua  indice_esgotamento_agua_consumida
             1101 MACRORREGIONAL II (CACOAL) 2017            True            32                      62.961447              5.886180                 13.957487                       40.232798                           5.886180
             1101 MACRORREGIONAL II (CACOAL) 2018            True            30                      64.284453              7.055335                 21.174387                       45.638420                           6.961049
             1101 MACRORREGIONAL II (CACOAL) 2019            True     

In [ ]:
# Formato no SNIS
print("=== SNIS ===")
print("dtype:", df_snis["id_municipio"].dtype)
print("exemplos:", df_snis["id_municipio"].head(5).tolist())
print("tamanho:", df_snis["id_municipio"].str.len().value_counts().to_dict())

# Formato na tabela de macrorregiões
print("\n=== Macrorregiões ===")
print("dtype:", df_macro["cod_municipio"].dtype)
print("exemplos:", df_macro["cod_municipio"].head(5).tolist())
print("tamanho:", df_macro["cod_municipio"].str.len().value_counts().to_dict())

=== SNIS ===
dtype: object
exemplos: ['1100015', '1100023', '1100031', '1100049', '1100056']
tamanho: {7: 42356}

=== Macrorregiões ===
dtype: string
exemplos: ['530010', '521880', '521190', '521310', '521850']
tamanho: {np.int64(6): 5571}


Os dados do SNIS foram obtidos via Base dos Dados (BigQuery), tabela br_mdr_snis.municipio_agua_esgoto, cobrindo 5.117 a 5.425 municípios por ano entre 2017-2022. Após merge com a tabela de macrorregiões de saúde (MS/Seidigi 2024) — convertendo o código IBGE de 7 dígitos do SNIS para 6 dígitos — obteve-se 0% de perda de registros. Os indicadores municipais foram agregados para macrorregião via média ponderada pela população estimada IBGE 2022. Para os anos de modelagem 2023-2024, ausentes do SNIS, foram criados registros por carry forward dos valores de 2022, estratégia sustentada pela variação interanual inferior a 1 p.p. observada no período 2017-2022.



In [ ]:
import pyarrow.parquet as pq
from pathlib import Path

DATA_INTERIM = Path('/content/drive/MyDrive/projeto_drsai/data/interim/sih_drsai')

# Pega uma amostra de 10 arquivos de anos diferentes
arquivos_amostra = []
for ano in [2017, 2019, 2023, 2024]:
    pasta_ano = DATA_INTERIM / str(ano)
    arquivos = list(pasta_ano.glob('*.parquet'))[:3]
    arquivos_amostra.extend(arquivos)

# Compara schemas
schemas = {}
for arq in arquivos_amostra:
    schema = pq.read_schema(arq)
    schemas[arq.name] = set(schema.names)

# Verifica se todos têm as mesmas colunas
schemas_list = list(schemas.values())
todos_iguais = all(s == schemas_list[0] for s in schemas_list)

if todos_iguais:
    print(f"✅ Schema consistente em todos os {len(arquivos_amostra)} arquivos testados")
    print(f"   Colunas: {sorted(schemas_list[0])}")
else:
    print("⚠️ Schemas inconsistentes!")
    for nome, cols in schemas.items():
        print(f"  {nome}: {sorted(cols)}")

✅ Schema consistente em todos os 12 arquivos testados
   Colunas: ['ANO_CMPT', 'COD_IDADE', 'DIAG_PRINC', 'DIAG_SECUN', 'DIAS_PERM', 'DT_INTER', 'DT_SAIDA', 'IDADE', 'MES_CMPT', 'MORTE', 'MUNIC_RES', 'N_AIH', 'RACA_COR', 'SEXO', 'UF_ZI', 'UTI_MES_TO', 'VAL_TOT', 'categoria_drsai']
